In [20]:
import sys
sys.path.append('.')
import samna
import numpy as np
import matplotlib

matplotlib.use("TkAgg")          # Or "Qt5Agg", "MacOSX", "WebAgg"
import matplotlib.pyplot as plt
import samna.dynapse1 as dyn1

import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator
from params_all_cores import *
import time
import importlib
from collections import deque
import threading

# sys.path.append('../Tools')

In [21]:
devices = samna.device.get_unopened_devices()
print(devices)

[]


In [3]:
#devices = samna.device.get_unopened_devices()
model   = samna.device.open_device(devices[int(0)])

In [4]:
api = model.get_dynapse1_api()
config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 
param_group_c1 = config1.chips[0].cores[1].parameter_group 
param_group_c2 = config1.chips[0].cores[2].parameter_group 
param_group_c3 = config1.chips[0].cores[3].parameter_group 

In [5]:
param_list = ["IF_AHTAU_N", "IF_AHTHR_N", "IF_AHW_P", "IF_BUF_P", "IF_DC_P", "IF_NMDA_N", "IF_RFR_N", "IF_TAU1_N", "IF_TAU2_N", "IF_THR_N", "NPDPIE_TAU_F_P", "NPDPIE_TAU_S_P", "NPDPIE_THR_F_P", "NPDPIE_THR_S_P", 
              "NPDPII_TAU_F_P", "NPDPII_TAU_S_P", "NPDPII_THR_F_P", "NPDPII_THR_S_P", "PS_WEIGHT_EXC_F_N", "PS_WEIGHT_EXC_S_N", "PS_WEIGHT_INH_F_N", "PS_WEIGHT_INH_S_N", "PULSE_PWLK_P", "R2R_P"]

In [6]:
# ----------------  stimulus: a Gaussian bump ----------------
n_pts     = 1000                 # number of samples
t_end     = 1.0                  # seconds  (→ dt = 1 ms)
t         = np.linspace(0, t_end, n_pts, endpoint=False)
x         = np.linspace(-4, 4, n_pts)
sigma = 0.6  # Try smaller values: 1.0 (default), 0.5, 0.25, etc.
gauss = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * (x / sigma)**2)
#I_peak    = 30000000e-12              # 1000 pA
I_peak    = 30000000e-12              # 1000 pA

I         = gauss/gauss.max() * I_peak   # injected current (A)

# convert to pA for nicer y‑axis numbers
I_pA = I * 1e12                 # A → pA

# ---------------- Plot stimulus waveform ----------------
plt.figure()
plt.plot(t*1e3, I_pA)           # x‑axis in ms
plt.xlabel('Time (ms)')
plt.ylabel('Injected current (pA)')
plt.title('Gaussian current stimulus (σ = 0.6)')
plt.tight_layout()

# ---------------- Plot spike times ----------------
# reproduce spike detection (same loop as user)
tau_m, R_m = 20e-3, 100e6
C_m = tau_m / R_m
v_rest = v_reset = -65e-3
v_thresh = -50e-3
t_ref  = 2e-3
dt     = t_end / n_pts


# ----------------  LIF neuron parameters ----------------------
tau_m     = 20e-3                # 20 ms membrane time constant
R_m       = 100e6                # 100 MΩ  (=> C = tau/R)
C_m       = tau_m / R_m
v_rest    = -65e-3               # -65 mV
v_reset   = -65e-3
v_thresh  = -50e-3               # spike threshold
t_ref     = 2e-3                 # 2 ms refractory period
dt        = t_end / n_pts        # simulation time-step (s)

# ----------------  simulation loop ----------------------------
v        = v_rest
next_ok  = 0.0                   # time when refractory ends
v_trace  = np.empty(n_pts)
spikes   = []

for k in range(n_pts):
    if t[k] >= next_ok:          # not in refractory
        dv = (-(v - v_rest) + R_m * I[k]) / (R_m * C_m) * dt
        v += dv
        if v >= v_thresh:        # spike!
            spikes.append(t[k])
            v = v_reset
            next_ok = t[k] + t_ref
    v_trace[k] = v

spike_times_all = np.array(spikes)

spike_ids = np.full(len(spikes), 1)

spikegen_ids = [(0, 1, n) for n in range(10)]

# separate figure for raster‑like spike markers
plt.figure()
plt.eventplot(spike_times_all*1e3, orientation='horizontal', linelength=0.1)
plt.xlabel('Time (ms)')
plt.yticks([])
plt.title(f'Spike times generated by the stimulus ({len(spike_times_all)} spikes)')
plt.tight_layout()

plt.show()

In [7]:
eventsBuffer = deque(maxlen=500)

In [8]:
def collect_spikes(sink_node, runningFlag):
    while runningFlag[0]:  # Check first element of list
        eventsBuffer.extend(sink_node.get_events())

In [9]:
import params_all_cores

importlib.reload(params_all_cores)
config1 = model.get_configuration()

pop_nr = 4

# 1)  Declare the set of bad neurons once, in (chip, core, neuron_id) format

BROKEN_NEURONS = {(0, 1, 51), (0, 1, 71), (0, 1, 80), (0, 1, 88), (0, 1, 91)}          #  ⬅️  add more here if needed

def is_ok(chip: int, core: int, nid: int) -> bool:
    """True if this physical neuron should be used."""
    return (chip, core, nid) not in BROKEN_NEURONS

In [10]:
spikegen_offset = 2

In [22]:
import importlib, dynapse1utils as ut

importlib.reload(params_all_cores)

importlib.reload(ut)

config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 

p_E_E   = 1
p_mexican = 1
p_I_I   = 0 #1
p_E_I   = 0 #0.1
p_I_E   = 0 #0.4

# initiate network 
net_gen = NetworkGenerator()
net_gen.clear_network()

# create spikegens, one per ring attractor neural pop 
spikegen_ids = [(0, 0, n+spikegen_offset) for n in range(10)]
#isi_spikegen = Neuron(0, 0, 200, True)
#isi_neuron = Neuron(0, 1, 200)

spikegens = []
for spikegen_id in spikegen_ids:
    spikegens.append(Neuron(spikegen_id[0], spikegen_id[1], spikegen_id[2], True))

print(spikegens)

#spikegens.append(isi_spikegen)

# Create ring neuron populations
chip = 0
core = 1
npop = 4
NBINS = 10
offset_nr = 48

ring_pops = []
next_id = offset_nr
for _ in range(NBINS):
    pop = []
    while len(pop) < npop:
        if is_ok(chip, core, next_id):
            pop.append(Neuron(chip, core, next_id))
        next_id += 1
    ring_pops.append(pop)
    
# Swap ring_pops[0] and ring_pops[1]
#ring_pops[0], ring_pops[5] = ring_pops[5], ring_pops[0]

# create inhibitory population that connects to all other pops
core_inh = 2
start_inh_neuron = 4
npop_inh = 4
pop_inhibitory = [Neuron(chip, core_inh, j) for j in range(start_inh_neuron, start_inh_neuron + npop_inh, 1)]
pop_inhibitory

# SPIKEGEN CONNECTIONS: one spike-gen (index i) permanently drives ring_pops[i]
for sg, pop in zip(spikegens, ring_pops):
    for neuron in pop:
        net_gen.add_connection(sg, neuron, dyn1.Dynapse1SynType.AMPA)
    
# connect isi spikegen to isi neuron 
#net_gen.add_connection(isi_spikegen, isi_neuron, dyn1.Dynapse1SynType.AMPA)

# self excitation in each neural population in the ring: (todo determine if this is needed) 
for pop in ring_pops:
    for pre in pop:
        for post in pop:
            if pre is not post and np.random.rand() < p_E_E:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)
                                
# MEXICAN HAT CONNECTIONS
OFFSET_1 = (-1, 1)         
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_1:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

OFFSET_2 = (-2, 2)          # ±3 bins wide “hat”

    # todo excitatory connections to second neighbors

for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_2:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)


    #  excitatory connections to third neighbors
OFFSET_3 = (-3, 3)          # ±3 bins wide “hat”
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_3:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                print(post)

# todo inhibitory connections to all of the other pops
OFFSET_inh = (-6, -5, -4, 4, 5, 6)          # all of the pops that are not being excited
for i, pop_i in enumerate(ring_pops):
    print(i,pop_i)
    for pre in pop_i:
        for d in OFFSET_inh:
            j = (i + d) % NBINS    
            print("j", j)# wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)

# INH → EXC  (global inhibition pop to all pops in the ring)
for inh in pop_inhibitory:
    for pop in ring_pops:
        for exc in pop:
            net_gen.add_connection(inh, exc, dyn1.Dynapse1SynType.GABA_B)

# EXC → INH  (drive the global inhibition pop from all pops in the ring)
for pop in ring_pops:
    for exc in pop:
        for inh in pop_inhibitory:
            net_gen.add_connection(exc, inh, dyn1.Dynapse1SynType.NMDA)

# make a dynapse1config using the network
new_config = net_gen.make_dynapse1_configuration()

# apply the configuration
model.apply_configuration(new_config)

print(net_gen.network)

# Set hardware parameters
set_params(model)

fpga_spike_gen = model.get_fpga_spike_gen() # set FPGA 


monitored_neurons = [
    (n.chip_id, n.core_id, n.neuron_id)
    for pop in ring_pops
    for n   in pop
]

monitored_neurons.extend([
    (neuron.chip_id, neuron.core_id, neuron.neuron_id)
    for neuron in pop_inhibitory  
])


graph, filter_node, sink_node = ut.create_neuron_select_graph(model, monitored_neurons)
graph.start()

# clear the buffer
sink_node.get_events()

# select the neurons to monitor
filter_node.set_neurons(monitored_neurons)

api.reset_timestamp()

ut.set_neuron_tau1(model, 0, 0, (7, 255))
ut.set_neuron_tau1(model, 0, 1, (7, 255))
ut.set_neuron_tau1(model, 0, 2, (7, 255))
ut.set_neuron_tau1(model, 0, 3, (7, 255))

time.sleep(1)

ut.set_neuron_tau1(model, 0, 0, (4, 50))
ut.set_neuron_tau1(model, 0, 1, (4, 50))
ut.set_neuron_tau1(model, 0, 2, (4, 50))
ut.set_neuron_tau1(model, 0, 3, (4, 50))


spike_ids_all = spike_ids

# Sort input events in time order
sort_indices = np.argsort(spike_times_all)
all_spike_times = spike_times_all[sort_indices]
all_spike_ids = spike_ids_all[sort_indices]

current_pop = {'value': 5}        # start with pop 0
spike_ids   = np.full(len(all_spike_times), current_pop['value'])

ut.set_fpga_spike_gen(
    fpga_spike_gen,
    all_spike_times,
    #all_spike_ids,
    spike_ids,
    #target_chips=[0] * len(all_spike_ids),
    target_chips=[0] * len(spike_ids),
    isi_base=900,
    repeat_mode=False)

fpga_spike_gen.start()

[C0c0s2, C0c0s3, C0c0s4, C0c0s5, C0c0s6, C0c0s7, C0c0s8, C0c0s9, C0c0s10, C0c0s11]
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n

Graph is destroyed while running! Note: Filter nodes constructed by `sequential` method won't work after corresponding graph is destroyed and please manually stop the graph after use.


In [12]:
print(ring_pops[0], ring_pops[5])


[C0c1n48, C0c1n49, C0c1n50, C0c1n52] [C0c1n69, C0c1n70, C0c1n72, C0c1n73]


In [13]:
ring_pops


[[C0c1n48, C0c1n49, C0c1n50, C0c1n52],
 [C0c1n53, C0c1n54, C0c1n55, C0c1n56],
 [C0c1n57, C0c1n58, C0c1n59, C0c1n60],
 [C0c1n61, C0c1n62, C0c1n63, C0c1n64],
 [C0c1n65, C0c1n66, C0c1n67, C0c1n68],
 [C0c1n69, C0c1n70, C0c1n72, C0c1n73],
 [C0c1n74, C0c1n75, C0c1n76, C0c1n77],
 [C0c1n78, C0c1n79, C0c1n81, C0c1n82],
 [C0c1n83, C0c1n84, C0c1n85, C0c1n86],
 [C0c1n87, C0c1n89, C0c1n90, C0c1n92]]

In [14]:
# ────────────────────────────────────── #
# ======== KDE-style smoothing =========
# ────────────────────────────────────── #
import scipy.ndimage as ndi

def smooth_rates_circular(counts, sigma_bins=1.0):
    """
    Gaussian-smooth an array living on a circular ring.
    counts      : 1-D numpy array of length NBINS
    sigma_bins  : std-dev of the Gaussian, expressed in *bin* units
    """
    # Pad three bins on each side so the wrap-around is seamless
    padded = np.r_[counts[-3:], counts, counts[:3]]
    smoothed = ndi.gaussian_filter1d(padded, sigma=sigma_bins, mode='wrap')
    return smoothed[3:-3]


In [15]:
import scipy.ndimage as ndi
def smooth_rates_circular(counts, sigma_bins=1.0):
    padded = np.r_[counts[-3:], counts, counts[:3]]
    smoothed = ndi.gaussian_filter1d(padded, sigma=sigma_bins, mode='wrap')

    return smoothed[3:-3]

In [17]:
# Start spike collection in a thread
running_flag = [True]  # Use list for mutable flag
spike_thread = threading.Thread(target=collect_spikes, args=(sink_node, running_flag))
spike_thread.daemon = True
spike_thread.start()
# animation = start_spike_visualization(eventsBuffer)

In [18]:
def on_key(event):
    k = event.key
    if k.isdigit():                       # 0-9 → choose new population
        new = int(k)
        if 0 <= new < NBINS:
            current_pop['value'] = new
            fpga_spike_gen.stop()         # reload the stimuli for that pop
            spike_ids[:] = new + spikegen_offset            # update *in-place* so length stays the same
            
            """ut.set_neuron_tau1(model, 0, 0, (7, 255))
            ut.set_neuron_tau1(model, 0, 1, (7, 255))
            ut.set_neuron_tau1(model, 0, 2, (7, 255))
            ut.set_neuron_tau1(model, 0, 3, (7, 255))

            time.sleep(1)

            ut.set_neuron_tau1(model, 0, 0, (4, 50))
            ut.set_neuron_tau1(model, 0, 1, (4, 50))
            ut.set_neuron_tau1(model, 0, 2, (4, 50))
            ut.set_neuron_tau1(model, 0, 3, (4, 50))"""
            
            ut.set_fpga_spike_gen(fpga_spike_gen,
                                   all_spike_times,
                                   spike_ids,
                                   target_chips=[0]*len(spike_ids),
                                   isi_base=900,
                                   repeat_mode=False)
            fpga_spike_gen.start()
            print(f"→ Stimulating pop {new}")
    elif k == 'v':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'h':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'q':                        # quit cleanly
        running_flag[0] = False

import matplotlib.colors as mcolors

# Create a mapping from neuron ID to ring population index and color
def create_neuron_to_pop_mapping(ring_pops):
    """Create mapping from neuron ID to population index"""
    neuron_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for neuron in pop:
            neuron_to_pop[neuron.neuron_id] = pop_idx
    return neuron_to_pop

# Create color map for populations
colors = plt.cm.tab10(np.linspace(0, 1, NBINS))  # Use tab10 colormap for distinct colors
neuron_to_pop = create_neuron_to_pop_mapping(ring_pops)

# Set up interactive plotting
plt.ion()

# Create figure with two subplots side by side
fig, (ax_raster, ax_rate) = plt.subplots(1, 2, figsize=(15, 6))
fig.canvas.mpl_connect('key_press_event', on_key)   # moved ↓ here
fig.show()                      # ← opens ONE browser tab

# Setup raster plot in first subplot
#scatter = ax_raster.scatter([], [], s=10, alpha=0.6)
xlim_max = 10
ax_raster.set_xlabel('Time (s)')
ax_raster.set_ylabel('Neuron Index')
ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

# Setup firing rate profile in second subplot
# Create positions for neurons (0 to 2π for the ring)
positions = np.linspace(0, 2*np.pi, NBINS, endpoint=False)

# dashed guide lines at every population centre
for p in positions:
    ax_rate.axvline(p,
                    linestyle='--',
                    linewidth=0.8,
                    color='gray',
                    alpha=0.4)

# optional: label each line with the pop index
for idx, p in enumerate(positions):
    ax_rate.text(p,            ax_rate.get_ylim()[1]*1.02,
                 str(idx),
                 ha='center', va='bottom', fontsize=9)

rate_line, = ax_rate.plot(positions, np.zeros_like(positions), 'o-', markersize=8)
ax_rate.set_xlim(0, 2*np.pi)
ax_rate.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax_rate.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax_rate.set_xlabel('Position (radians)')
ax_rate.set_ylabel('Firing rate (Hz)')
ax_rate.set_title('Firing Rate Profile')
ax_rate.grid(True, alpha=0.3)

plt.tight_layout()

# Variables for rate calculation
window_size = 1.0  # seconds - time window for calculating rates
last_update_time = 0

# Real-time plotting loop
while running_flag[0]:
    try:
        if len(eventsBuffer) > 0:
            # Extract spike data for raster plot
            spikesID = [e.neuron_id for e in eventsBuffer]
            spikesTimes = [e.timestamp*1e-6 for e in eventsBuffer]  # Convert to seconds
            
            print(spikesTimes)
            print(spikesID)
            
            if not spikesID:
                continue
                
            # Update raster plot
            """ax_raster.set_ylim(min(spikesID) - 0.5, max(spikesID) + 0.5)
            scatter.set_offsets(np.column_stack((spikesTimes, spikesID)))"""
            
            # Clear previous raster plot
            ax_raster.clear()
            ax_raster.set_xlabel('Time (s)')
            ax_raster.set_ylabel('Neuron Index')
            ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

            # Group spikes by population and plot with different colors
            for pop_idx in range(NBINS):
                # Find spikes belonging to this population
                pop_spike_times = []
                pop_spike_ids = []
                
                for spike_time, spike_id in zip(spikesTimes, spikesID):
                    if spike_id in neuron_to_pop and neuron_to_pop[spike_id] == pop_idx:
                        pop_spike_times.append(spike_time)
                        pop_spike_ids.append(spike_id)
                
                # Plot spikes for this population with its assigned color
                if pop_spike_times:  # Only plot if there are spikes
                    ax_raster.scatter(pop_spike_times, pop_spike_ids, 
                                    s=10, alpha=0.8, 
                                    color=colors[pop_idx], 
                                    label=f'Pop {pop_idx}')

            # Add legend (optional - you can remove if it clutters)
            ax_raster.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            
            # Update time window
            current_time = max(spikesTimes)
            ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
            ax_raster.set_ylim(0, 95)
            
            # Calculate firing rates across the ring (using recent time window)
            window_start = current_time - window_size
            recent_events = [e for e in eventsBuffer if e.timestamp*1e-6 >= window_start]
            
            # Count spikes for each bin in the ring
            firing_rates = np.zeros(NBINS)
            
            # Map neuron IDs to their bin/position in the ring
            """for event in recent_events:
                neuron_id = event.neuron_id
                for i, pop in enumerate(ring_pops):
                    pop_ids = [n.neuron_id for n in pop]
                    if neuron_id in pop_ids:
                        firing_rates[i] += 1
                        break"""
            # Map neuron IDs to their bin/position in the ring
            for event in recent_events:
                neuron_id = event.neuron_id
                if neuron_id in neuron_to_pop:
                    pop_idx = neuron_to_pop[neuron_id]
                    firing_rates[pop_idx] += 1
                    
            # numpy histogram for firing rate then divide by bins
            
            # Convert to Hz (spikes per second)
            firing_rates = firing_rates / window_size
            
            # Update firing rate plot
            smoothed_rates = smooth_rates_circular(firing_rates, sigma_bins=1.0)
            rate_line.set_ydata(smoothed_rates)
            max_rate = max(smoothed_rates) if any(smoothed_rates > 0) else 10
            
            ax_rate.set_ylim(0, 40)  

            
            # Refresh both plots
            fig.canvas.draw_idle()
            fig.canvas.flush_events()   # keeps the websocket alive
            time.sleep(0.01)            # tiny CPU-friendly sleep

            #plt.pause(0.0001)  # Shorter pause for smoother updates
            
    except Exception as e:
        print(f"Plotting error: {e}")
        import traceback
        traceback.print_exc()  # Print detailed error information
        break

/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_36122/1043674055.py:108: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


[1.300791, 1.389997, 1.421682, 1.491638, 1.5245229999999999, 1.6545159999999999, 1.6947189999999999, 1.738745, 1.749835, 1.76288, 1.784009, 1.793226, 1.805269, 1.8194359999999998, 1.8371579999999998, 1.901335, 1.969212, 1.9845849999999998, 1.991595, 2.0232829999999997, 2.067654, 2.069954, 2.100949, 2.118495, 2.151401, 2.209856, 2.217991, 2.228523, 2.234787, 2.244751, 2.266133, 2.2735659999999998, 2.2880059999999998, 2.345249, 2.366608, 2.3702609999999997, 2.392818, 2.408696, 2.4234709999999997, 2.473137, 2.476288, 2.494083, 2.5053389999999998, 2.506231, 2.558795, 2.57725, 2.597912, 2.655522, 2.73232, 2.7446669999999997, 2.83676, 2.838685, 2.892479, 2.89335, 2.96913, 2.989916, 3.002671, 3.0205859999999998, 3.033095, 3.0621069999999997, 3.1470089999999997, 3.1562959999999998, 3.171474, 3.1766959999999997, 3.208135, 3.2286539999999997, 3.251319, 3.3739909999999997, 3.3826959999999997, 3.5202269999999998, 3.525845, 3.56996, 3.582191, 3.6526189999999996, 3.668237, 3.673289, 3.70026599999999

In [19]:
# Create a colormap for the ring populations
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

def on_key(event):
    k = event.key
    if k.isdigit():                       # 0-9 → choose new population
        new = int(k)
        if 0 <= new < NBINS:
            current_pop['value'] = new
            fpga_spike_gen.stop()         # reload the stimuli for that pop
            spike_ids[:] = new            # update *in-place* so length stays the same
            
            """ut.set_neuron_tau1(model, 0, 0, (7, 255))
            ut.set_neuron_tau1(model, 0, 1, (7, 255))
            ut.set_neuron_tau1(model, 0, 2, (7, 255))
            ut.set_neuron_tau1(model, 0, 3, (7, 255))

            time.sleep(1)

            ut.set_neuron_tau1(model, 0, 0, (4, 50))
            ut.set_neuron_tau1(model, 0, 1, (4, 50))
            ut.set_neuron_tau1(model, 0, 2, (4, 50))
            ut.set_neuron_tau1(model, 0, 3, (4, 50))"""
            
            ut.set_fpga_spike_gen(fpga_spike_gen,
                                   all_spike_times,
                                   spike_ids,
                                   target_chips=[0]*len(spike_ids),
                                   isi_base=900,
                                   repeat_mode=False)
            fpga_spike_gen.start()
            print(f"→ Stimulating pop {new}")
    elif k == 'v':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'h':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'q':                        # quit cleanly
        running_flag[0] = False


# Create a mapping from neuron ID to ring population index and color
def create_neuron_to_pop_mapping(ring_pops):
    """Create mapping from neuron ID to population index"""
    neuron_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for neuron in pop:
            neuron_to_pop[neuron.neuron_id] = pop_idx
    return neuron_to_pop

# Create color map for populations
colors = plt.cm.tab10(np.linspace(0, 1, NBINS))  # Use tab10 colormap for distinct colors
neuron_to_pop = create_neuron_to_pop_mapping(ring_pops)

# Set up interactive plotting
plt.ion()

# Create figure with two subplots side by side
fig, (ax_raster, ax_rate) = plt.subplots(1, 2, figsize=(15, 6))
fig.canvas.mpl_connect('key_press_event', on_key)
fig.show()

# Setup raster plot in first subplot - Remove the old scatter plot
ax_raster.set_xlabel('Time (s)')
ax_raster.set_ylabel('Neuron Index')
ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

# Setup firing rate profile in second subplot
positions = np.linspace(0, 2*np.pi, NBINS, endpoint=False)

# dashed guide lines at every population centre
for p in positions:
    ax_rate.axvline(p,
                    linestyle='--',
                    linewidth=0.8,
                    color='gray',
                    alpha=0.4)

# optional: label each line with the pop index
for idx, p in enumerate(positions):
    ax_rate.text(p, ax_rate.get_ylim()[1]*1.02,
                 str(idx),
                 ha='center', va='bottom', fontsize=9)

rate_line, = ax_rate.plot(positions, np.zeros_like(positions), 'o-', markersize=8)
ax_rate.set_xlim(0, 2*np.pi)
ax_rate.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax_rate.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax_rate.set_xlabel('Position (radians)')
ax_rate.set_ylabel('Firing rate (Hz)')
ax_rate.set_title('Firing Rate Profile')
ax_rate.grid(True, alpha=0.3)

plt.tight_layout()

# Variables for rate calculation
window_size = 1.0
last_update_time = 0

# Real-time plotting loop
while running_flag[0]:
    try:
        if len(eventsBuffer) > 0:
            # Extract spike data for raster plot
            spikesID = [e.neuron_id for e in eventsBuffer]
            spikesTimes = [e.timestamp*1e-6 for e in eventsBuffer]
            
            print(spikesTimes)
            print(spikesID)
            
            if not spikesID:
                continue
            
            # Clear previous raster plot
            ax_raster.clear()
            ax_raster.set_xlabel('Time (s)')
            ax_raster.set_ylabel('Neuron Index')
            ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')
            
            # Group spikes by population and plot with different colors
            for pop_idx in range(NBINS):
                # Find spikes belonging to this population
                pop_spike_times = []
                pop_spike_ids = []
                
                for spike_time, spike_id in zip(spikesTimes, spikesID):
                    if spike_id in neuron_to_pop and neuron_to_pop[spike_id] == pop_idx:
                        pop_spike_times.append(spike_time)
                        pop_spike_ids.append(spike_id)
                
                # Plot spikes for this population with its assigned color
                if pop_spike_times:  # Only plot if there are spikes
                    ax_raster.scatter(pop_spike_times, pop_spike_ids, 
                                    s=10, alpha=0.8, 
                                    color=colors[pop_idx], 
                                    label=f'Pop {pop_idx}')
            
            # Update time window
            current_time = max(spikesTimes)
            ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
            ax_raster.set_ylim(0, 95)
            
            # Add legend (optional - you can remove if it clutters)
            ax_raster.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            
            # Calculate firing rates across the ring (using recent time window)
            window_start = current_time - window_size
            recent_events = [e for e in eventsBuffer if e.timestamp*1e-6 >= window_start]
            
            # Count spikes for each bin in the ring
            firing_rates = np.zeros(NBINS)
            
            # Map neuron IDs to their bin/position in the ring
            for event in recent_events:
                neuron_id = event.neuron_id
                if neuron_id in neuron_to_pop:
                    pop_idx = neuron_to_pop[neuron_id]
                    firing_rates[pop_idx] += 1
            
            # Convert to Hz (spikes per second)
            firing_rates = firing_rates / window_size
            
            # Update firing rate plot
            smoothed_rates = smooth_rates_circular(firing_rates, sigma_bins=1.0)
            rate_line.set_ydata(smoothed_rates)
            max_rate = max(smoothed_rates) if any(smoothed_rates > 0) else 10
            
            ax_rate.set_ylim(0, 40)
            
            # Refresh both plots
            fig.canvas.draw_idle()
            fig.canvas.flush_events()
            time.sleep(0.01)
            
    except Exception as e:
        print(f"Plotting error: {e}")
        import traceback
        traceback.print_exc()
        break

[22.897956999999998, 22.913856, 22.924360999999998, 23.091918999999997, 23.096731, 23.106911, 23.119248, 23.169173, 23.18209, 23.200848, 23.22955, 23.234574, 23.273062, 23.293135, 23.295704, 23.301887, 23.395063999999998, 23.401536, 23.452025, 23.472361, 23.484227, 23.523502999999998, 23.527931, 23.536932, 23.585378, 23.671436, 23.714472999999998, 23.715889, 23.721103, 23.818073, 23.82761, 23.870635999999998, 23.888855, 23.891135, 23.969631, 24.001542999999998, 24.012435, 24.026815, 24.034354999999998, 24.110035, 24.144185999999998, 24.157732, 24.179164, 24.201722, 24.214724, 24.228400999999998, 24.263216999999997, 24.333343, 24.372562, 24.40945, 24.454756, 24.491854999999997, 24.516631999999998, 24.524251, 24.540307, 24.552173, 24.556745, 24.618664, 24.6343, 24.666522, 24.695940999999998, 24.708534999999998, 24.749126999999998, 24.799455, 24.821832999999998, 24.840816999999998, 24.889733999999997, 24.950219999999998, 25.002722, 25.028052, 25.03595, 25.062134, 25.078367, 25.127616, 25.

/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_32776/2292087932.py:109: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
Traceback (most recent call last):
  File "/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_32776/2292087932.py", line 155, in <module>
    ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
                                             ^^^^^^^^
NameError: name 'xlim_max' is not defined


In [ ]:
# To stop everything cleanly:
running_flag[0] = False
spike_thread.join(timeout=1.0)
plt.ioff()
plt.close()

In [34]:
# ===== OFFLINE SIMULATION WITH VELOCITY MODULATION =====

print("Starting offline simulation sequence...")

time_phase1 = 1.0
time_phase2 = 12.0
time_phase3 = 5.0
# Initialize storage for all events
all_events = []
simulation_phases = []  # Track which phase each event belongs to

# Clear any existing events
eventsBuffer.clear()
sink_node.get_events()

# === PHASE 1: Initial stimulation of population 5 for 2 seconds ===
print(f"Phase 1: Stimulating population 5 for {time_phase1} seconds...")

# Set FPGA to stimulate population 5
spike_ids[:] = 5
ut.set_fpga_spike_gen(fpga_spike_gen,
                      all_spike_times, spike_ids,
                      target_chips=[0]*len(spike_ids),
                      isi_base=900, repeat_mode=False)

# Start simulation
fpga_spike_gen.start()
phase1_start = time.time()

while time.time() - phase1_start < time_phase1:
    new_events = sink_node.get_events()
    for event in new_events:
        all_events.append(event)
        simulation_phases.append("Phase 1: Initial")
    time.sleep(0.01)

fpga_spike_gen.stop()
print(f"Phase 1 complete. Collected {len([p for p in simulation_phases if p == 'Phase 1: Initial'])} events.")

# === PHASE 2: Add velocity connections and run for 7 seconds ===
print(f"Phase 2: Adding velocity connections and running for {time_phase2} seconds...")

# Add velocity connections (equivalent to 'v' key press)
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        j = (i + 1) % NBINS  # wrap around
        for post in ring_pops[j]:
            net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

# Apply new configuration
config = net_gen.make_dynapse1_configuration()
model.apply_configuration(config)
print("Velocity connections added.")

# Continue stimulation
# fpga_spike_gen.start()
phase2_start = time.time()

while time.time() - phase2_start < time_phase2:
    new_events = sink_node.get_events()
    for event in new_events:
        all_events.append(event)
        simulation_phases.append("Phase 2: Velocity")
    time.sleep(0.01)

# fpga_spike_gen.stop()
print(f"Phase 2 complete. Collected {len([p for p in simulation_phases if p == 'Phase 2: Velocity'])} events.")

# === PHASE 3: Remove velocity connections and run for 3 seconds ===
print(f"Phase 3: Removing velocity connections and running for {time_phase3} seconds...")

# Remove velocity connections (equivalent to 'h' key press)
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        j = (i + 1) % NBINS  # wrap around
        for post in ring_pops[j]:
            net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

# Apply new configuration
config = net_gen.make_dynapse1_configuration()
model.apply_configuration(config)
print("Velocity connections removed.")

# Continue stimulation
# fpga_spike_gen.start()
phase3_start = time.time()

while time.time() - phase3_start < time_phase3:
    new_events = sink_node.get_events()
    for event in new_events:
        all_events.append(event)
        simulation_phases.append("Phase 3: No velocity")
    time.sleep(0.01)

# fpga_spike_gen.stop()
print(f"Phase 3 complete. Collected {len([p for p in simulation_phases if p == 'Phase 3: No velocity'])} events.")

# Final event collection
final_events = sink_node.get_events()
for event in final_events:
    all_events.append(event)
    simulation_phases.append("Phase 3: No velocity")

print(f"Simulation complete! Total events collected: {len(all_events)}")


"""# === END OF SIMULATION === #"""

"""# === PLOTTING PART === #"""

# === RASTER PLOT RESULTS ===
print("Creating raster plot...")

if len(all_events) > 0:
    # Extract event data and normalize timestamps to start from 0
    raw_timestamps = [e.timestamp * 1e-6 for e in all_events]  # Convert to seconds
    start_time = min(raw_timestamps)  # Get the first timestamp
    event_times = [t - start_time for t in raw_timestamps]  # Normalize to start from 0
    event_neuron_ids = [e.neuron_id for e in all_events]
    
    # Create neuron ID to population mapping for coloring
    nid_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for n in pop:
            nid_to_pop[n.neuron_id] = pop_idx
    
    # Give inhibitory population its own index
    INH_IDX = NBINS
    for n in pop_inhibitory:
        nid_to_pop[n.neuron_id] = INH_IDX
    
    # Map colors
    import matplotlib.cm as cm
    pop_colors = cm.get_cmap('tab10', NBINS+1)
    evt_colors = []
    for nid in event_neuron_ids:
        pop_idx = nid_to_pop.get(nid, None)
        if pop_idx is None or pop_idx == INH_IDX:
            evt_colors.append('lightgrey')
        else:
            evt_colors.append(pop_colors(pop_idx))
    
    # Create the plot
    plt.figure(figsize=(15, 8))
    plt.scatter(event_times, event_neuron_ids, s=3, c=evt_colors, alpha=0.7)
    
    # Add vertical lines to mark phase transitions
    plt.axvline(time_phase1, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Velocity ON')
    plt.axvline(time_phase1 + time_phase2, color='blue', linestyle='--', linewidth=2, alpha=0.8, label='Velocity OFF') 
    
    # Add population labels
    for pop in range(NBINS):
        ids = [n.neuron_id for n in ring_pops[pop]]
        y_lim =max(ids)
        # if ids:
        #     y_pos = np.mean(ids)
        #     plt.text(-0.5, y_pos, f'P{pop}', va='center', ha='right', fontsize=8, 
        #             color=pop_colors(pop), weight='bold')
    
    plt.ylim(0,y_lim+5)
    plt.xlabel('Time (s)', fontsize=12)
    plt.ylabel('Neuron ID', fontsize=12)
    plt.title(f'Offline Simulation: Population 5 Stimulation with Velocity Modulation\n' + 
              f'Phase 1 (0-{time_phase1}s): Initial | Phase 2 ({time_phase1}-{time_phase1+time_phase2}s): +Velocity | Phase 3 ({time_phase1+time_phase2}-{time_phase1+time_phase2+time_phase3}s): -Velocity', 
              fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    # Print summary statistics
    phase1_events = len([p for p in simulation_phases if p == 'Phase 1: Initial'])
    phase2_events = len([p for p in simulation_phases if p == 'Phase 2: Velocity']) 
    phase3_events = len([p for p in simulation_phases if p == 'Phase 3: No velocity'])
    
    print(f"\nSimulation Summary:")
    print(f"Phase 1 (Initial, 0-{time_phase1}s): {phase1_events} events")
    print(f"Phase 2 (Velocity, {time_phase1}-{time_phase1+time_phase2}s): {phase2_events} events") 
    print(f"Phase 3 (No velocity, {time_phase1+time_phase2}-{time_phase1+time_phase2+time_phase3}s): {phase3_events} events")
    print(f"Total events: {len(all_events)}")
    print(f"Total duration: {time_phase1+time_phase2+time_phase3} seconds")
    
    plt.tight_layout()
    # plt.show()
    
    
    
    # === PEAK TRACKING ANALYSIS ===
    print("\nStarting peak tracking analysis...")
    
    from scipy.optimize import minimize
    from scipy.stats import circmean
    
    def circular_gaussian(x, mu, sigma, A):
        
        # Handle periodic boundary conditions
        diff = np.array([(xi - mu + NBINS/2) % NBINS - NBINS/2 for xi in x])
        return A * np.exp(-0.5 * (diff / sigma)**2)
    
    def fit_circular_gaussian(firing_rates):
        positions = np.arange(NBINS)
        
        # Skip if no activity
        if np.sum(firing_rates) == 0:
            return np.nan, np.nan, np.nan  # mu, sigma, A
        
        # Initial guess: peak at maximum firing rate position
        max_pos = np.argmax(firing_rates)
        initial_A = np.max(firing_rates)
        initial_sigma = 1.0
        
        def objective(params):
            mu, sigma, A = params
            if sigma <= 0 or A < 0:
                return 1e10
            predicted = circular_gaussian(positions, mu, sigma, A)
            return np.sum((firing_rates - predicted)**2)
        
        # Try optimization with different initial conditions
        best_result = None
        best_error = np.inf
        
        for init_mu in [max_pos, (max_pos + 1) % NBINS, (max_pos - 1) % NBINS]:
            try:
                result = minimize(objective, [init_mu, initial_sigma, initial_A],
                                method='L-BFGS-B',
                                bounds=[(0, NBINS-1), (0.1, NBINS/2), (0, None)])
                if result.success and result.fun < best_error:
                    best_result = result
                    best_error = result.fun
            except:
                continue
        
        if best_result is not None:
            mu, sigma, A = best_result.x
            # Normalize mu to [0, NBINS) range
            mu = mu % NBINS
            return mu, sigma, A
        else:
            return np.nan, np.nan, np.nan
    
    # Analysis parameters
    timestep = 0.1  # seconds
    total_duration = time_phase1 + time_phase2 + time_phase3  # seconds
    time_bins = np.arange(0, total_duration + timestep, timestep)
    
    # Storage for results
    peak_positions = []
    peak_times = []
    all_firing_rates = []
    fit_quality = []
    
    print(f"Analyzing {len(time_bins)-1} time windows...")
    
    for i in range(len(time_bins) - 1):
        t_start = time_bins[i]
        t_end = time_bins[i + 1]
        
        # Find events in this time window
        window_events = []
        for j, event_time in enumerate(event_times):
            if t_start <= event_time < t_end:
                window_events.append((event_time, event_neuron_ids[j]))
        
        # Calculate firing rates for each population
        firing_rates = np.zeros(NBINS)
        for event_time, neuron_id in window_events:
            if neuron_id in nid_to_pop and nid_to_pop[neuron_id] < NBINS:  # Exclude inhibitory
                pop_idx = nid_to_pop[neuron_id]
                firing_rates[pop_idx] += 1
        
        # Convert to Hz (events per second)
        firing_rates = firing_rates / timestep
        all_firing_rates.append(firing_rates.copy())
        
        # Fit circular Gaussian
        mu, sigma, A = fit_circular_gaussian(firing_rates)
        
        if not np.isnan(mu):
            peak_positions.append(mu)
            peak_times.append(t_start + timestep/2)  # Center of time window
            
            # Calculate fit quality (R-squared)
            predicted = circular_gaussian(np.arange(NBINS), mu, sigma, A)
            ss_res = np.sum((firing_rates - predicted)**2)
            ss_tot = np.sum((firing_rates - np.mean(firing_rates))**2)
            r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            fit_quality.append(r_squared)
        else:
            fit_quality.append(0)
    
    peak_positions = np.array(peak_positions)
    peak_times = np.array(peak_times)
    fit_quality = np.array(fit_quality)
    
    # Convert peak positions to degrees (360° / 10 bins = 36° per bin)
    peak_positions_degrees = peak_positions * 360.0 / NBINS
    
    # Create enhanced peak tracking plot
    plt.figure(figsize=(12, 6))
    plt.plot(peak_times, peak_positions_degrees, 'ro-', markersize=4, linewidth=1.5, alpha=0.8, label='Peak Position')
    
    # Add vertical lines to mark phase transitions
    plt.axvline(time_phase1, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Velocity ON')
    plt.axvline(time_phase1 + time_phase2, color='blue', linestyle='--', linewidth=2, alpha=0.8, label='Velocity OFF')
    
    # Styling
    plt.xlabel('Time (s)', fontsize=12)
    plt.ylabel('Peak Position (degrees)', fontsize=12)
    plt.title('Ring Attractor Peak Position Over Time\n(Population Activity Center Tracking)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=10)
    
    # Set y-axis to show full circle range with nice tick marks
    plt.ylim(0, 360)
    plt.yticks(np.arange(0, 361, 45), [f'{int(deg)}°' for deg in np.arange(0, 361, 45)])
    
    # Add horizontal reference lines for easier reading
    for deg in np.arange(0, 361, 90):
        plt.axhline(deg, color='gray', linestyle=':', alpha=0.2, linewidth=0.8)
    
    plt.tight_layout()
    plt.show()

    print(f"Successfully tracked {len(peak_positions)} peaks out of {len(time_bins)-1} time windows")
    
    # Convert peak positions to radians for plotting
    # peak_positions_rad = peak_positions * 2 * np.pi / NBINS
    
    
    """
    # === PLOT PEAK TRAJECTORY ===
    plt.figure(figsize=(15, 10))
    
    # Top plot: Peak position over time
    plt.subplot(3, 1, 1)
    if len(peak_positions) > 0:
        plt.plot(peak_times, peak_positions_rad, 'bo-', markersize=4, linewidth=1.5, alpha=0.8)
        plt.axvline(2.0, color='red', linestyle='--', alpha=0.7, label='Velocity ON')
        plt.axvline(9.0, color='blue', linestyle='--', alpha=0.7, label='Velocity OFF')
    plt.ylabel('Peak Position (rad)')
    plt.title('Ring Attractor Peak Tracking Analysis')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.ylim(0, 2*np.pi)
    plt.yticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi], ['0', 'π/2', 'π', '3π/2', '2π'])
    
    # Middle plot: Peak position in population space
    plt.subplot(3, 1, 2)
    if len(peak_positions) > 0:
        plt.plot(peak_times, peak_positions, 'ro-', markersize=4, linewidth=1.5, alpha=0.8)
        plt.axvline(2.0, color='red', linestyle='--', alpha=0.7)
        plt.axvline(9.0, color='blue', linestyle='--', alpha=0.7)
    plt.ylabel('Peak Position (Population #)')
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.5, NBINS-0.5)
    plt.yticks(range(NBINS))
    
    # Bottom plot: Fit quality over time
    plt.subplot(3, 1, 3)
    quality_times = [t_start + timestep/2 for t_start in time_bins[:-1]]
    plt.plot(quality_times, fit_quality, 'go-', markersize=3, linewidth=1, alpha=0.8)
    plt.axvline(2.0, color='red', linestyle='--', alpha=0.7)
    plt.axvline(9.0, color='blue', linestyle='--', alpha=0.7)
    plt.xlabel('Time (s)')
    plt.ylabel('Fit Quality (R²)')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    # === VELOCITY ANALYSIS ===
    if len(peak_positions) > 1:
        # Calculate angular velocity (handling wrap-around)
        velocities = []
        velocity_times = []
        
        for i in range(1, len(peak_positions)):
            dt = peak_times[i] - peak_times[i-1]
            
            # Handle circular differences
            pos_diff = peak_positions[i] - peak_positions[i-1]
            if pos_diff > NBINS/2:
                pos_diff -= NBINS
            elif pos_diff < -NBINS/2:
                pos_diff += NBINS
                
            # Convert to angular velocity (rad/s)
            angular_velocity = pos_diff * 2 * np.pi / NBINS / dt
            velocities.append(angular_velocity)
            velocity_times.append((peak_times[i] + peak_times[i-1]) / 2)
        
        velocities = np.array(velocities)
        velocity_times = np.array(velocity_times)
        
        # Plot velocity
        plt.figure(figsize=(12, 6))
        plt.plot(velocity_times, velocities, 'mo-', markersize=4, linewidth=1.5, alpha=0.8)
        plt.axvline(2.0, color='red', linestyle='--', alpha=0.7, label='Velocity ON')
        plt.axvline(9.0, color='blue', linestyle='--', alpha=0.7, label='Velocity OFF')
        plt.axhline(0, color='black', linestyle='-', alpha=0.3)
        plt.xlabel('Time (s)')
        plt.ylabel('Angular Velocity (rad/s)')
        plt.title('Ring Attractor Angular Velocity')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print(f"\nPeak Tracking Statistics:")
        print(f"Total tracked peaks: {len(peak_positions)}")
        print(f"Mean fit quality (R²): {np.mean(fit_quality):.3f}")
        print(f"Peak position range: {np.min(peak_positions):.1f} to {np.max(peak_positions):.1f}")
        
        if len(velocities) > 0:
            # Phase-specific velocity analysis
            phase1_vel = velocities[(velocity_times >= 0) & (velocity_times < 2)]
            phase2_vel = velocities[(velocity_times >= 2) & (velocity_times < 9)]
            phase3_vel = velocities[(velocity_times >= 9) & (velocity_times < 12)]
            
            print(f"\nVelocity Analysis:")
            if len(phase1_vel) > 0:
                print(f"Phase 1 (Initial): Mean velocity = {np.mean(phase1_vel):.3f} rad/s")
            if len(phase2_vel) > 0:
                print(f"Phase 2 (Velocity): Mean velocity = {np.mean(phase2_vel):.3f} rad/s")
            if len(phase3_vel) > 0:
                print(f"Phase 3 (No velocity): Mean velocity = {np.mean(phase3_vel):.3f} rad/s")
    
else:
    print("No events collected during simulation!")"""

Starting offline simulation sequence...
Phase 1: Stimulating population 5 for 1.0 seconds...
Phase 1 complete. Collected 20 events.
Phase 2: Adding velocity connections and running for 12.0 seconds...
Phase 1 complete. Collected 20 events.
Phase 2: Adding velocity connections and running for 12.0 seconds...
Velocity connections added.
Velocity connections added.
Phase 2 complete. Collected 434 events.
Phase 3: Removing velocity connections and running for 5.0 seconds...
Phase 2 complete. Collected 434 events.
Phase 3: Removing velocity connections and running for 5.0 seconds...
Velocity connections removed.
Velocity connections removed.
Phase 3 complete. Collected 176 events.
Simulation complete! Total events collected: 631
Creating raster plot...

Simulation Summary:
Phase 1 (Initial, 0-1.0s): 20 events
Phase 2 (Velocity, 1.0-13.0s): 434 events
Phase 3 (No velocity, 13.0-18.0s): 177 events
Total events: 631
Total duration: 18.0 seconds

Starting peak tracking analysis...
Analyzing 180

/tmp/ipykernel_7737/1930038622.py:134: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  pop_colors = cm.get_cmap('tab10', NBINS+1)


Successfully tracked 164 peaks out of 180 time windows
